# Phân Loại Ảnh Khớp Gối (Kellgren-Lawrence Grade) Sử Dụng PyTorch và TIMM

Notebook này chứa toàn bộ mã nguồn để huấn luyện một mô hình phân loại ảnh chụp X-quang khớp gối thành 5 cấp độ theo thang đo Kellgren-Lawrence (KL grades):
* `0Normal` (Bình thường)
* `1Doubtful` (Nghi ngờ)
* `2Mild` (Nhẹ)
* `3Moderate` (Trung bình)
* `4Severe` (Nặng)

Mô hình sử dụng kiến trúc **ResNet18** (hoặc tùy chọn các kiến trúc khác từ thư viện `timm`) với trọng số được huấn luyện trước (pretrained) trên ImageNet.

In [1]:
%pip install timm scikit-learn matplotlib tqdm pillow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 38.3 MB/s eta 0:00:00a 0:00:01


### BƯỚC 2: Kết nối (Mount) với Google Drive để nạp dữ liệu

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### BƯỚC 3: Định nghĩa Dataset và Dataloader

In [3]:
import os
import glob
import json
import random
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
from PIL import Image
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

class KneeXRayDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]
        
        if self.transform:
            image = self.transform(image)
            
        return image, label

def get_transforms(img_size=224):
    train_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    return train_transform, val_transform

def get_image_paths_and_labels(data_dirs):
    classes = ['0Normal', '1Doubtful', '2Mild', '3Moderate', '4Severe']
    class_to_idx = {cls: idx for idx, cls in enumerate(classes)}
    
    image_paths = []
    labels = []
    
    if isinstance(data_dirs, str):
        data_dirs = [data_dirs]
        
    for data_dir in data_dirs:
        data_dir = os.path.abspath(os.path.expanduser(data_dir))
        for cls in classes:
            cls_dir = os.path.join(data_dir, cls)
            if not os.path.isdir(cls_dir):
                continue
            for ext in ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG'):
                files = glob.glob(os.path.join(cls_dir, ext))
                for f in files:
                    image_paths.append(f)
                    labels.append(class_to_idx[cls])
                    
    return image_paths, labels, classes

def create_dataloaders(data_dirs, batch_size=16, img_size=224, val_split=0.2, num_workers=2, random_state=42):
    image_paths, labels, classes = get_image_paths_and_labels(data_dirs)
    
    if len(image_paths) == 0:
        raise ValueError(f"Không tìm thấy ảnh nào. Hãy chắc chắn đường dẫn đúng và chứa các thư mục con: {classes}")
        
    print(f"[Dataset] Tổng số ảnh: {len(image_paths)}")
    for cls_name in classes:
        cls_idx = classes.index(cls_name)
        print(f"  - Lớp '{cls_name}': {labels.count(cls_idx)} ảnh")
        
    train_paths, val_paths, train_labels, val_labels = train_test_split(
        image_paths, labels, test_size=val_split, stratify=labels, random_state=random_state
    )
    
    train_transform, val_transform = get_transforms(img_size)
    train_dataset = KneeXRayDataset(train_paths, train_labels, transform=train_transform)
    val_dataset = KneeXRayDataset(val_paths, val_labels, transform=val_transform)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
    
    print(f"[Dataset] Train: {len(train_dataset)} | Val: {len(val_dataset)}")
    return train_loader, val_loader, classes

### BƯỚC 4: Xây dựng kiến trúc mô hình (TIMM)

In [4]:
import timm

def build_model(model_name='resnet18', num_classes=5, pretrained=True):
    print(f"[Model] Đang khởi tạo mô hình {model_name} (num_classes={num_classes})")
    model = timm.create_model(model_name, pretrained=pretrained, num_classes=num_classes)
    return model

### BƯỚC 5: Thiết lập các hàm Huấn luyện và Đánh giá

In [5]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    progress_bar = tqdm(dataloader, desc="  Training", leave=False)
    for images, labels in progress_bar:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
    return running_loss / total, 100. * correct / total

def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        progress_bar = tqdm(dataloader, desc="  Validating", leave=False)
        for images, labels in progress_bar:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
    return running_loss / total, 100. * correct / total

### BƯỚC 6: Bắt đầu Huấn luyện

*Hãy thay đổi đường dẫn trong danh sách `DATA_DIRS` bên dưới cho chính xác với vị trí lưu các thư mục ảnh trên Google Drive của bạn.*

In [6]:
# --- CẤU HÌNH THÔNG SỐ HUẤN LUYỆN ---
DATA_DIRS = [
    "/content/drive/MyDrive/dicom/Digital Knee X-ray Images/Knee X-ray Images/MedicalExpert-I",
    "/content/drive/MyDrive/dicom/Digital Knee X-ray Images/Knee X-ray Images/MedicalExpert-II"
]
OUTPUT_DIR = "./outputs"
MODEL_NAME = "resnet18"
EPOCHS = 10
BATCH_SIZE = 16
LR = 1e-4
IMG_SIZE = 224

os.makedirs(OUTPUT_DIR, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Thiết bị đang sử dụng: {device}")

# 1. Khởi tạo DataLoaders
train_loader, val_loader, class_names = create_dataloaders(
    data_dirs=DATA_DIRS,
    batch_size=BATCH_SIZE,
    img_size=IMG_SIZE,
    num_workers=2
)

# Lưu tên các lớp
class_mapping_path = os.path.join(OUTPUT_DIR, "class_names.json")
with open(class_mapping_path, 'w', encoding='utf-8') as f:
    json.dump(class_names, f, ensure_ascii=False, indent=4)

# 2. Khởi tạo mô hình
model = build_model(model_name=MODEL_NAME, num_classes=len(class_names), pretrained=True)
model = model.to(device)

# 3. Bộ tối ưu hóa và Scheduler
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

# 4. Vòng lặp huấn luyện chính
best_acc = 0.0
print("\n--- BẮT ĐẦU QUÁ TRÌNH HUẤN LUYỆN ---")
for epoch in range(1, EPOCHS + 1):
    print(f"\nEpoch {epoch}/{EPOCHS}")
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    
    scheduler.step()
    
    print(f"  -> Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"  -> Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")
    
    if val_acc > best_acc:
        best_acc = val_acc
        best_model_path = os.path.join(OUTPUT_DIR, "best_model.pth")
        torch.save(model.state_dict(), best_model_path)
        print(f"  [Save] Đã lưu checkpoint tốt nhất tại: {best_model_path} ({val_acc:.2f}%)")
        
print(f"\nHUẤN LUYỆN THÀNH CÔNG! Độ chính xác tốt nhất trên Val: {best_acc:.2f}%")

Thiết bị đang sử dụng: cpu
[Dataset] Tổng số ảnh: 3300
  - Lớp '0Normal': 1017 ảnh
  - Lớp '1Doubtful': 965 ảnh
  - Lớp '2Mild': 464 ảnh
  - Lớp '3Moderate': 442 ảnh
  - Lớp '4Severe': 412 ảnh
[Dataset] Train: 2640 | Val: 660
[Model] Đang khởi tạo mô hình resnet18 (num_classes=5)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]


--- BẮT ĐẦU QUÁ TRÌNH HUẤN LUYỆN ---

Epoch 1/10


  Training:   0%|          | 0/165 [00:00<?, ?it/s]

...

  Validating:   0%|          | 0/42 [00:00<?, ?it/s]

  -> Train Loss: 0.6487 | Train Acc: 76.25%
  -> Val Loss: 0.5403 | Val Acc: 79.70%
  [Save] Đã lưu checkpoint tốt nhất tại: ./outputs/best_model.pth (79.70%)

Epoch 10/10


  Training:   0%|          | 0/165 [00:00<?, ?it/s]

  Validating:   0%|          | 0/42 [00:00<?, ?it/s]

  -> Train Loss: 0.6350 | Train Acc: 76.14%
  -> Val Loss: 0.5474 | Val Acc: 79.09%

HUẤN LUYỆN THÀNH CÔNG! Độ chính xác tốt nhất trên Val: 79.70%


### BƯỚC 7: Dự đoán trên ảnh X-Quang mới

In [13]:
def predict_single_image(image_path, model_path="./outputs/best_model.pth", class_names_path="./outputs/class_names.json", model_name="resnet18"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    with open(class_names_path, 'r', encoding='utf-8') as f:
        class_names = json.load(f)
        
    model = build_model(model_name=model_name, num_classes=len(class_names), pretrained=False)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model = model.to(device)
    model.eval()
    
    _, val_transform = get_transforms(224)
    image = Image.open(image_path).convert('RGB')
    input_tensor = val_transform(image).unsqueeze(0).to(device)
    
    with torch.no_grad():
        outputs = model(input_tensor)
        probabilities = F.softmax(outputs, dim=1)[0]
        
    best_class_idx = torch.argmax(probabilities).item()
    best_class = class_names[best_class_idx]
    best_prob = probabilities[best_class_idx].item()
    
    print("\n--- KẾT QUẢ DỰ ĐOÁN ---")
    print(f"Đường dẫn ảnh: {image_path}")
    print(f"Dự đoán: {best_class} ({best_prob * 100:.2f}%)")
    print("Chi tiết xác suất từng phân lớp:")
    for i, prob in enumerate(probabilities):
        print(f"  - {class_names[i]}: {prob * 100:.2f}%")

# Bạn có thể bỏ comment dòng dưới đây và truyền đường dẫn ảnh của bạn để chạy dự đoán thử:
predict_single_image(
    image_path="/content/drive/MyDrive/dicom/Digital Knee X-ray Images/Knee X-ray Images/MedicalExpert-II/3Moderate/ModerateG3 (13).png",
    model_path="./outputs/best_model.pth",
    class_names_path="./outputs/class_names.json"
)

[Model] Đang khởi tạo mô hình resnet18 (num_classes=5)

--- KẾT QUẢ DỰ ĐOÁN ---
Đường dẫn ảnh: /content/drive/MyDrive/dicom/Digital Knee X-ray Images/Knee X-ray Images/MedicalExpert-II/3Moderate/ModerateG3 (13).png
Dự đoán: 3Moderate (86.46%)
Chi tiết xác suất từng phân lớp:
  - 0Normal: 0.25%
  - 1Doubtful: 0.70%
  - 2Mild: 6.30%
  - 3Moderate: 86.46%
  - 4Severe: 6.28%
